# ComplianceRadar: fine-tune the event classifier with LoRA

Trains a small multilingual encoder (`xlm-roberta-base`) with **LoRA (PEFT)** to label regulatory/news headlines with one of 13 topics.

- **Training data:** LLM-distilled headlines (`data/classifier/train.jsonl`, made by `scripts/make_classifier_dataset.py`).
- **Test data:** 52 hand-written English headlines (`data/fixtures/classifier_test.jsonl`) the generator never saw.
- **Baselines on the same test set:** keyword rules 0.60 accuracy; rules → LLM 0.92 with 24 LLM calls.
- **Result of the original run (13 Nov 2026):** 0.808 accuracy alone; as the middle tier (rules → LoRA → LLM) 0.94 with 5 LLM calls. That run's adapter is in `models/event-classifier-lora/`; its library versions other than PEFT 0.21.2 were not recorded.

**Tested stack (14 Nov 2026):** transformers 5.22.2, peft 0.21.2, accelerate 1.15.0, datasets 5.2.0, scikit-learn 1.9.1, torch 2.15. The cells were executed end to end with this stack on CPU as a short smoke run (2 training steps; `scripts/smoke_test_notebook.py`, record in `artifacts/notebook_smoke_2026-11-14.json`). That checks the code runs, saves and reloads with the same labels; it is not a full training run. Transformers 5 removed `warmup_ratio`; a float `warmup_steps` below 1 is the ratio now.

Runtime: **GPU (T4 is enough)**, about 5 minutes. The last cells write `run_manifest.json` (versions, seed, data sizes, scores, file hashes) and check that the saved adapter reloads with the same label mapping and predictions.

In [ ]:
# The stack these cells were tested with (see the first cell). Colab's preinstalled torch is used as is.
!pip -q install "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.2.0" "scikit-learn==1.9.1"
# Colab ships an old torchao that newer peft refuses to import alongside. This notebook does not use torchao.
!pip -q uninstall -y torchao

In [ ]:
# Get the repo (for the data files). Change the URL if your fork differs.
!git clone -q https://github.com/Muskaanmahadik10/compliance-radar.git || (cd compliance-radar && git pull -q)
%cd compliance-radar
!wc -l data/classifier/train.jsonl data/fixtures/classifier_test.jsonl

In [ ]:
import json
import random

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows = load("data/classifier/train.jsonl")
test_rows = load("data/fixtures/classifier_test.jsonl")
labels = sorted({r["topic"] for r in train_rows} | {r["topic"] for r in test_rows})
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}

# Leakage guard: no test headline may appear in training.
test_titles = {r["title"].lower() for r in test_rows}
train_rows = [r for r in train_rows if r["title"].lower() not in test_titles]
random.shuffle(train_rows)
n_val = max(len(labels) * 4, int(0.1 * len(train_rows)))
val_rows, train_rows = train_rows[:n_val], train_rows[n_val:]
print(len(train_rows), "train /", len(val_rows), "validation /", len(test_rows), "test ·", len(labels), "labels")

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

BASE = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(BASE)

def to_ds(rows):
    ds = Dataset.from_list([{"text": r["title"], "label": label2id[r["topic"]]} for r in rows])
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

ds_train, ds_val, ds_test = to_ds(train_rows), to_ds(val_rows), to_ds(test_rows)

model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(labels), id2label=id2label, label2id=label2id)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "value"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # LoRA trains well under 1.5% of the weights

In [ ]:
def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pred), "macro_f1": f1_score(p.label_ids, pred, average="macro")}

args = TrainingArguments(
    output_dir="clf-out", learning_rate=5e-4, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    # warmup_steps=0.1 is a 10% warmup: transformers 5 removed warmup_ratio and reads a float < 1 as a ratio.
    num_train_epochs=12, weight_decay=0.01, warmup_steps=0.1, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1", logging_steps=20, seed=SEED, report_to="none",
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val, data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# Final check on the hand-written test set (never seen in training or model selection).
out = trainer.predict(ds_test)
pred = out.predictions.argmax(-1)
print("TEST accuracy %.3f · macro-F1 %.3f" % (accuracy_score(out.label_ids, pred), f1_score(out.label_ids, pred, average="macro")))
print(classification_report(out.label_ids, pred, target_names=labels, zero_division=0))
for r, p in zip(test_rows, pred):
    if id2label[int(p)] != r["topic"]:
        print("MISS:", r["topic"], "->", id2label[int(p)], "|", r["title"])

result = {"tier": "lora (xlm-roberta-base, r=16)", "n": len(test_rows), "accuracy": round(float(accuracy_score(out.label_ids, pred)), 3),
          "macro_f1": round(float(f1_score(out.label_ids, pred, average="macro")), 3), "train_rows": len(train_rows), "cost_usd_per_1k_events": 0.0}
json.dump(result, open("lora_classifier_result.json", "w"), indent=2); result

## Save and (optionally) publish the adapter

Paste the numbers from `lora_classifier_result.json` into the README table. To use the model in the worker, push the adapter to the Hub and set `RADAR_CLASSIFIER_MODEL=<your-username>/compliance-radar-event-classifier`.

In [ ]:
model.save_pretrained("clf-adapter"); tok.save_pretrained("clf-adapter")
# The label order is part of the model: save it next to the adapter so a loader never has to reconstruct it.
json.dump({"num_labels": len(labels), "id2label": {str(k): v for k, v in id2label.items()}, "label2id": label2id},
          open("clf-adapter/label_mapping.json", "w"), indent=2)
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("YOUR_USERNAME/compliance-radar-event-classifier"); tok.push_to_hub("YOUR_USERNAME/compliance-radar-event-classifier")

In [ ]:
# Run manifest: what produced this adapter, so the run can be reproduced and the files checked later.
import hashlib, platform, subprocess, sys
import accelerate, datasets, peft, sklearn, transformers

def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

manifest = {
    "seed": SEED, "base_model": BASE, "lora": {"r": 16, "alpha": 32, "dropout": 0.1, "target_modules": ["query", "value"]},
    "data": {"train": len(train_rows), "validation": len(val_rows), "test": len(test_rows), "labels": len(labels),
             "train_sha256": sha256("data/classifier/train.jsonl"), "test_sha256": sha256("data/fixtures/classifier_test.jsonl")},
    "repo_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
                 "accelerate": accelerate.__version__, "datasets": datasets.__version__, "scikit-learn": sklearn.__version__},
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "training_args": {k: getattr(args, k) for k in ("learning_rate", "num_train_epochs", "max_steps", "warmup_steps", "per_device_train_batch_size", "weight_decay")},
    "result": result,
    "files_sha256": {f: sha256(f"clf-adapter/{f}") for f in ("adapter_model.safetensors", "adapter_config.json", "label_mapping.json")},
}
json.dump(manifest, open("clf-adapter/run_manifest.json", "w"), indent=2); manifest["versions"]

In [ ]:
# Reload check: base model + saved adapter + saved label mapping must give the same labels and the same predictions.
from peft import PeftModel

saved = json.load(open("clf-adapter/label_mapping.json"))
id2 = {int(k): v for k, v in saved["id2label"].items()}
assert id2 == id2label and saved["label2id"] == label2id, "label mapping changed between training and saving"
base = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(id2), id2label=id2, label2id=saved["label2id"])
reloaded = PeftModel.from_pretrained(base, "clf-adapter").eval()
model.eval()
enc = tok([r["title"] for r in test_rows], truncation=True, max_length=64, padding=True, return_tensors="pt")
with torch.no_grad():
    dev = next(model.parameters()).device
    a = model(**{k: v.to(dev) for k, v in enc.items()}).logits.argmax(-1).cpu()
    b = reloaded(**enc).logits.argmax(-1)
assert torch.equal(a, b), "reloaded adapter predicts differently"
print("reload OK:", len(id2), "labels,", int((a == b).sum()), "/", len(test_rows), "identical predictions")